# SVG2 stages 1–2 by hand: from a raw video to object tracks with SAM2

This notebook builds the first two stages of the SVG2 data pipeline on a real video, one step at a time, with a picture at every step:

| stage | what it does | knows about time? |
|---|---|---|
| **1. maskgen** | SAM2 "segment everything" on single frames, plus the 90% filter | **no**: every frame is on its own |
| **2. track** | SAM2's *video* predictor carries each object through time with one fixed id, and finds objects that appear later | **yes** |

SAM2 itself is only used as the black box "mask in → same object in the next frames out". Everything around it, meaning which frames, when new objects are added, matching and the two passes, is written out here. Each part is checked against their real code (`assert`s and a full re-run of their `stage2_track` at the end).

**Where it is in the paper:**
- §3.1 "Phase 1" (p. 5).
- Appendix A.1, pp. 20–23: the 90% filter on p. 20, *online tracking* Eqs. 6–13 on pp. 21–22, *offline tracking* Eq. 14 on pp. 22–23.
- Table 7 (p. 20) for the settings.

**Where it is in the code** (`third_party/SVG2/pipeline/svg2_pipeline.py`, commit 77caa44):

| function | line |
|---|---|
| `stage1_generate_masks` | 1370 |
| `stage2_track` | 1399 |
| `_find_new_objects` | 1491 |
| `match_detections_to_tracks` | 524 |
| `VideoTracker` | 721 |

**Two places where the code and the paper may disagree, tested below:**
1. **When is a new object added?** The paper (Eqs. 9–11) compares the *union of all* stage-1 masks with the untracked area. The code requires *each* new mask on its own to cover ≥ 10% of the untracked area.
2. **Does the offline pass really fill in frames before an object was found?** The paper (p. 22) says so. The code only propagates forward from frame 0. We measure it.

**Run:** GPU on. Run the cells top to bottom. With the defaults (5 s of video, SAM2-large) expect very roughly 20–40 min on a T4; most of that is tracking. Set `RUN_OFFICIAL_CHECK = False` to skip the last re-run.

In [ ]:
# ---- setup: works both in VS Code (repo already there) and in a fresh Kaggle notebook ----
import os, subprocess, sys, time, json
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"            # model weights go to the big temporary disk
r = subprocess.run("bash setup_kaggle.sh", shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])          # look for lines starting with "!!"
sys.path.insert(0, os.path.join(REPO, "third_party/SVG2/pipeline"))

In [ ]:
import cv2, numpy as np, torch, shutil, colorsys
import matplotlib.pyplot as plt
from contextlib import nullcontext
import svg2_pipeline as P                           # THEIR pipeline code
import sam2
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "|", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "(no GPU: turn it on!)")
print("sam2 from", os.path.dirname(sam2.__file__))

## Settings

The defaults are the paper's settings (Table 7 = `pipeline/examples/run_full.yaml`). The only change is that we use the first 5 seconds of the video so that it runs in reasonable time.

In [ ]:
VIDEO = "third_party/SVG2/pipeline/examples/sav_000001.mp4"   # the example video shipped with their code
MAX_FRAMES = 120                    # first 5 s (24 fps). None = whole video (483 frames, ~4x slower)
MODEL = P.DEFAULT_SAM2_MODEL_ID     # facebook/sam2.1-hiera-large, as in the paper (both stages)
STAGE1_GRIDS = [32, 16, 4]          # Table 7. [32] = no zoomed crops: much faster, misses small things
RUN_OFFICIAL_CHECK = True           # at the end, re-run THEIR stage2_track and compare with ours

OUT = "notes/stage12"; os.makedirs(OUT, exist_ok=True)       # pictures + README (push these)
CLIP_DIR = "/tmp/clip_jpg"                                    # the clip as JPEG frames, for SAM2
mcfg = P.MaskGenConfig(model_id=MODEL, grid_points_per_side=STAGE1_GRIDS)   # their stage-1 settings
tcfg = P.TrackingConfig(model_id=MODEL)                                     # their stage-2 settings
for k, v in vars(tcfg).items():
    print(f"  tracking.{k} = {v}")

LOG = []                            # everything said with say() also goes into notes/stage12/README.md
def say(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

In [ ]:
# ---- drawing helpers ----
def colour(i):
    """A fixed colour per object id, so the same object has the same colour in every frame."""
    return np.array(colorsys.hsv_to_rgb((i * 0.618034) % 1.0, 0.75, 1.0))

def overlay(frame, masks, alpha=0.5, label=True):
    """masks = {id: bool HxW}. Paints each mask in its id's colour and writes the id on it."""
    img = frame.astype(np.float32) / 255
    for i, m in masks.items():
        img[m] = (1 - alpha) * img[m] + alpha * colour(i)
    img = (img * 255).astype(np.uint8)
    if label:
        for i, m in masks.items():
            ys, xs = np.nonzero(m)
            if len(xs):
                p = (int(xs.mean()) - 6, int(ys.mean()) + 5)
                cv2.putText(img, str(i), p, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3)
                cv2.putText(img, str(i), p, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return img

def show(images, titles=None, cols=3, size=4.5, save=None, suptitle=None):
    rows = int(np.ceil(len(images) / cols))
    h, w = images[0].shape[:2]
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size * h / w + 0.6), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for k, im in enumerate(images):
        ax = axes.ravel()[k]
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        if titles:
            ax.set_title(titles[k], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    if save:
        fig.savefig(f"{OUT}/{save}", dpi=70)
    plt.show()

def timer():
    t0 = time.time()
    return lambda: f"{time.time() - t0:.0f} s"

## 1. The input: a video is just a stack of images

Their reader (`P.read_video_frames`) decodes every frame to an RGB array of shape (H, W, 3). That is all the pipeline starts from: no labels, no boxes, no ids.

SAM2's video predictor reads either a `.mp4` or a folder of `00000.jpg, 00001.jpg, …`. To use only the first `MAX_FRAMES` we write them out as JPEGs. We then read them back so that stage 1 and stage 2 see the same pixels.

In [ ]:
frames_all, fps, H, W = P.read_video_frames(VIDEO)
T = len(frames_all) if MAX_FRAMES is None else min(MAX_FRAMES, len(frames_all))
say(f"Video `{os.path.basename(VIDEO)}`: {len(frames_all)} frames, {fps:.0f} fps, {W}x{H}, {len(frames_all) / fps:.1f} s. "
    f"We use the first {T} frames ({T / fps:.1f} s).")
shutil.rmtree(CLIP_DIR, ignore_errors=True); os.makedirs(CLIP_DIR)
for i in range(T):
    cv2.imwrite(f"{CLIP_DIR}/{i:05d}.jpg", cv2.cvtColor(frames_all[i], cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
frames = [cv2.cvtColor(cv2.imread(f"{CLIP_DIR}/{i:05d}.jpg"), cv2.COLOR_BGR2RGB) for i in range(T)]
del frames_all
print("one frame:", frames[0].shape, frames[0].dtype, "→ the clip is an array of shape", (T,) + frames[0].shape)
idx = np.linspace(0, T - 1, 6).astype(int)
show([frames[i] for i in idx], [f"frame {i}  (t = {i / fps:.1f} s)" for i in idx], save="1_filmstrip.png",
     suptitle="The input: a stack of images. Nothing says which pixels belong to which object.")

## 2. The schedule: which frames get looked at

Two different frame lists matter:

- **Stage 1 frames.** "Segment everything" runs on **every 20th frame** (`frame_sample_rate = 20`). It is far too slow to run on every frame.
- **Re-discovery checks.** Stage 2 only looks for *new* objects at the stage-1 frames that are multiples of `20 × (T // 100 + 1)`. That gap grows with video length. The paper (p. 22) only says breakpoints are "evaluated periodically".

The code reads the stage-1 result of only two kinds of frame: the **first** stage-1 frame (the seed) and the **check** frames. Their pipeline still runs stage 1 on all the other every-20th frames, but stage 2 never reads those results. So here we run stage 1 only on the frames that are used. Stage 2's output is identical either way, as long as the first frame has any masks.

In [ ]:
RATE = mcfg.frame_sample_rate                                   # 20
sampled = list(range(0, T, RATE))                               # where their stage 1 runs
check_stride = RATE * (T // 100 + 1) if tcfg.adaptive_sample_rate else RATE
check = sorted(f for f in sampled if f % check_stride == 0)     # where stage 2 may add new objects
seed = sampled[0]                                               # all stage-1 masks of this frame become objects
needed = sorted(set(check) | {seed})
say(f"Stage 1 frames (every {RATE}th): {sampled}")
say(f"Re-discovery stride = {RATE} × ({T} // 100 + 1) = {check_stride} frames → checks at {check} (the seed frame {seed} itself is skipped)")
say(f"Stage-1 results stage 2 actually reads: {needed}  ({len(needed)} of {len(sampled)})")
for n in (120, 483, 1000, 3000):
    s = list(range(0, n, RATE)); st = RATE * (n // 100 + 1); c = [f for f in s if f % st == 0]
    say(f"  a {n}-frame video: stage 1 on {len(s)} frames, {len(c)} used; a new object can only be added every "
        f"{st} frames = {st / 24:.1f} s at 24 fps")

fig, ax = plt.subplots(figsize=(12, 1.4))
ax.scatter(range(T), [0] * T, s=2, c="lightgray", label="frame")
ax.scatter(sampled, [0] * len(sampled), s=40, c="tab:blue", label="stage 1 (every 20th)")
ax.scatter(check, [0] * len(check), s=160, facecolors="none", edgecolors="tab:red", linewidths=2, label="re-discovery check")
ax.set_yticks([]); ax.set_xlabel("frame")
ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.8))
fig.savefig(f"{OUT}/2_schedule.png", dpi=70, bbox_inches="tight"); plt.show()

## 3. Stage 1 on those frames ("segment everything")

This is the step we built by hand in `tools/stage1_by_hand.py`: a 32×32 point grid, plus 16×16 and 4×4 grids on zoomed crops, then filters, then the 90% filter (p. 20). Here we call their code directly.

**Look at the numbers on the masks.** They are only positions in a list, sorted by size. Mask #3 in frame 0 and mask #3 in frame 40 are *not* known to be the same thing. Stage 1 has **no notion of time**: it hands stage 2 a separate bag of masks for each frame.

In [ ]:
t = timer()
gen = P.AutomaticMaskGenerator(mcfg, dev)
stage1 = {}
for f in needed:
    raw = gen.generate(frames[f])
    stage1[f] = P.select_max_non_overlapping(raw, mcfg.max_overlap_ratio)
    say(f"Stage 1, frame {f}: {len(raw)} SAM2 masks → {len(stage1[f])} after the 90% filter ({t()})")
del gen; torch.cuda.empty_cache()
show([overlay(frames[f], dict(enumerate(stage1[f]))) for f in needed],
     [f"frame {f}: {len(stage1[f])} masks, numbered 0…{len(stage1[f]) - 1}" for f in needed],
     cols=min(3, len(needed)), save="3_stage1.png",
     suptitle="Stage 1: a separate bag of masks per frame. The numbers mean nothing across frames.")

Could we just link masks across frames by overlap? That only works if nothing moves, and the frames are 40 frames (1.7 s) apart. Below, each of the 5 largest masks of the first frame is matched with its best-overlapping mask in the next frame:

In [ ]:
if len(needed) > 1:
    a, b = needed[0], needed[1]
    for i in np.argsort([-m.sum() for m in stage1[a]])[:5]:
        ious = [(stage1[a][i] & m).sum() / (stage1[a][i] | m).sum() for m in stage1[b]]
        j = int(np.argmax(ious))
        print(f"frame {a} mask #{i} ({stage1[a][i].sum()} px)  ↔  frame {b} mask #{j}: IoU {ious[j]:.2f}")

## 4. SAM2's video predictor on ONE object: where the "temporal connection" comes from

The image predictor answers "what is at this point *in this image*". The video predictor answers "**where is this object now**, given what it looked like before". Its calls:

1. **`init_state(folder)`**: loads all frames, resized to 1024×1024. Frames are encoded later, one at a time, during tracking.
2. **`add_new_mask(state, frame, obj_id, mask)`**: the *prompt*. SAM2 encodes "this frame + this mask" into a **memory** for object `obj_id`. This is called a *conditioning* frame.
3. **`propagate_in_video(state, start, n)`**: for every next frame, and **for every object separately**:
   1. Encode the frame (once per frame, shared by all objects).
   2. **Memory attention.** The frame's features "look at" this object's memories: its prompted frame(s) and its own masks from the **last 6 frames**. They also look at short "object pointer" vectors from earlier frames. In effect: "find the thing that looked like *this*".
   3. The mask decoder outputs the mask, plus an "is the object visible at all?" score. If the answer is no, the mask is **empty** (occluded or gone).
   4. The new mask is encoded into a new memory for the following frames.

The identity lives in each object's **memory**, not in pixel positions, so the object keeps its id while it moves. The cost grows linearly with the number of objects, which is one reason the pipeline stops at 65 (`max_objects`).

Below: one stage-1 mask from the first frame is given as id 0 and propagated through the clip.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor_hf
t = timer()
predictor = build_sam2_video_predictor_hf(MODEL, device=dev)
state = predictor.init_state(CLIP_DIR, offload_video_to_cpu=tcfg.offload_to_cpu, offload_state_to_cpu=tcfg.offload_to_cpu)
print(f"video predictor loaded, {state['num_frames']} frames resized to {predictor.image_size}x{predictor.image_size} ({t()})")
print(f"memory per object: its prompted frame(s) + the last {predictor.num_maskmem - 1} frames it tracked")

def propagate(state, start, max_frames):
    """Exactly their VideoTracker.propagate (l.754): yields (frame, {object id: bool mask}), going forward."""
    ac = torch.autocast("cuda", dtype=torch.bfloat16) if dev == "cuda" else nullcontext()
    with torch.inference_mode(), ac:
        for f, ids, logits in predictor.propagate_in_video(state, start_frame_idx=start, max_frame_num_to_track=max_frames):
            binary = (logits > 0.0).cpu().numpy()
            if binary.ndim == 4:
                binary = binary[:, 0]
            yield int(f), {int(o): binary[k] for k, o in enumerate(ids)}

In [ ]:
frac = [m.sum() / (H * W) for m in stage1[seed]]
# the biggest mask under 15% of the frame: likely a "thing" rather than floor, wall or sky
demo = next((i for i in np.argsort(frac)[::-1] if frac[i] < 0.15), int(np.argmax(frac)))
predictor.add_new_mask(state, seed, 0, stage1[seed][demo].astype(np.uint8))
t = timer(); demo_track = {}
for f, tracked in propagate(state, seed, T - seed - 1):
    demo_track[f] = tracked[0]
say(f"One object (stage-1 mask #{demo} of frame {seed}) tracked through {len(demo_track)} frames in {t()}")
idx = np.linspace(seed, T - 1, 6).astype(int)
show([overlay(frames[f], {0: demo_track[f]}) for f in idx], [f"frame {f}: {demo_track[f].sum()} px" for f in idx],
     save="4_one_object.png", suptitle="ONE mask given at the first frame → the same object (id 0) found in every later frame")
fr = sorted(demo_track)
plt.figure(figsize=(10, 2.5)); plt.plot(fr, [demo_track[f].sum() for f in fr])
plt.xlabel("frame"); plt.ylabel("mask area (px)"); plt.title("Object 0's area over time (0 = SAM2 says it is not visible)")
plt.savefig(f"{OUT}/4_one_object_area.png", dpi=70, bbox_inches="tight"); plt.show()
predictor.reset_state(state)        # forget this demo object

## 5. Pass 1, "online tracking": track everything and discover new objects (paper pp. 21–22)

1. **Seed:** every stage-1 mask of the first frame becomes an object with ids 0, 1, 2, … This is recorded in `first_appearance` (the paper's 𝓡, Eq. 6).
2. **Propagate** forward. At each **check frame** (red circles in step 2):
   - **Coverage** C_t = union of all tracked masks (Eq. 8). **Untracked** U_t = everything else. If U_t < 5% of the frame, skip the check.
   - For each stage-1 mask at this frame, ask: **is it an object we already track?** Compute how much of the *detection* lies inside each tracked mask (Eq. 12, asymmetric). If ≥ 90% lies inside one tracked object, it **is** that object (Eq. 13).
   - Otherwise it is a candidate new object. **Code:** it is added only if *it alone* covers ≥ 10% of U_t. **Paper (Eqs. 9–11):** the *union of all* stage-1 masks is compared with U_t. We print both.
   - If anything is new: **stop**, register the new masks with new ids at this frame, and **restart propagation from the next frame**.
3. Stop adding objects once 65 are tracked.

Each check is drawn with three panels:
- what the tracker has;
- the untracked region (white);
- the stage-1 detections: **green** = already tracked, **red** = new and added, **orange** = new but ignored for being too small.

In [ ]:
def find_new_objects_by_hand(tracked, dets):
    """Their _find_new_objects (l.1491) + match_detections_to_tracks (l.524), written out,
    and also returning WHY each detection was kept or not."""
    covered = np.zeros((H, W), bool)
    for m in tracked.values():
        covered |= m                                           # C_t, Eq. 8
    untracked = ~covered                                       # U_t
    U = int(untracked.sum())
    info = {"untracked_frac": U / (H * W), "dets": []}
    if U / (H * W) < tcfg.rediscovery_min_untracked_frac:      # < 5% untracked: nothing to look for
        info["skipped"] = True
        return [], info
    new = []
    for d in dets:
        A = int(d.sum())
        r = {i: int((d & m).sum()) / A for i, m in tracked.items()}   # Eq. 12: share of the DETECTION inside object i
        best = max(r, key=r.get) if r else None
        if best is not None and r[best] >= tcfg.match_iou_thresh:     # Eq. 13: ≥ 0.9 → it is object `best`
            verdict = f"already tracked (= object {best}, {r[best]:.0%} inside it)"
        else:
            share = int((d & untracked).sum()) / U                     # code: THIS mask alone vs U_t
            if share >= tcfg.rediscovery_overlap_thresh:
                verdict = f"NEW → added (covers {share:.0%} of the untracked area)"
                new.append(d)
            else:
                verdict = f"not matched, but covers only {share:.1%} of the untracked area (< 10%) → ignored"
        info["dets"].append((d, verdict))
    D = np.zeros((H, W), bool)
    for d in dets:
        D |= d                                                 # paper Eq. 9: union of ALL detections
    info["paper_ratio"] = int((D & untracked).sum()) / U       # paper Eq. 11 (trigger if ≥ 0.1)
    return new, info

In [ ]:
t = timer()
first_appearance = {}               # 𝓡: id -> (frame where it was registered, its mask there)
next_id = 0
for m in stage1[seed]:
    predictor.add_new_mask(state, seed, next_id, m.astype(np.uint8))
    first_appearance[next_id] = (seed, m); next_id += 1
say(f"Pass 1: seeded {next_id} objects at frame {seed}")

pass1 = {}                          # frame -> {id: RLE mask}: pass 1's latest answer per frame
checks_log = []
start = seed
while start < T:
    pending = None
    for f, tracked in propagate(state, start, T - start - 1):
        pass1[f] = {i: P.encode_rle(m) for i, m in tracked.items() if m.any()}
        if not (tcfg.rediscovery_enabled and f in check and f > seed and next_id < tcfg.max_objects):
            continue
        dets = stage1.get(f, [])
        if not dets:
            continue
        new, info = find_new_objects_by_hand(tracked, dets)
        theirs = P._find_new_objects(tracked, dets, H, W, tcfg)              # their function must agree
        assert len(theirs) == len(new) and all((a == b).all() for a, b in zip(theirs, new)), "differs from their code!"
        checks_log.append((f, {i: m for i, m in tracked.items() if m.any()}, info))
        say(f"Pass 1, check at frame {f}: untracked {info['untracked_frac']:.0%} of the frame; {len(dets)} stage-1 masks → "
            f"{len(new)} new (code rule). Paper Eq. 11 ratio, all masks together: {info.get('paper_ratio', float('nan')):.2f}")
        if new:
            pending = (f, new)
            break                   # stop here, register, restart from the next frame
    if pending is None:
        break
    f, new = pending
    new = new[: tcfg.max_objects - next_id]
    for m in new:
        predictor.add_new_mask(state, f, next_id, m.astype(np.uint8))
        first_appearance[next_id] = (f, m); next_id += 1
    say(f"   → registered objects {next_id - len(new)}…{next_id - 1} at frame {f}; propagation restarts at frame {f + 1}")
    start = f + 1
say(f"Pass 1 done: {next_id} objects ({t()})")

In [ ]:
cols = {"NEW": (1, 0, 0), "not": (1, 0.6, 0), "already": (0, 0.8, 0)}
for f, tracked, info in checks_log:
    if info.get("skipped"):
        print(f"frame {f}: skipped, only {info['untracked_frac']:.1%} untracked"); continue
    covered = np.zeros((H, W), bool)
    for m in tracked.values():
        covered |= m
    det = frames[f].astype(np.float32) / 255
    for d, v in info["dets"]:
        det[d] = 0.45 * det[d] + 0.55 * np.array(cols[v.split()[0]])
    show([overlay(frames[f], tracked), (~covered).astype(np.uint8) * 255, (det * 255).astype(np.uint8)],
         [f"frame {f}: what the tracker has ({len(tracked)} objects)",
          f"untracked region U_t (white) = {info['untracked_frac']:.0%} of the frame",
          "stage-1 masks: green = already tracked, red = NEW, orange = ignored"],
         save=f"5_check_frame{f}.png")
    for k, (d, v) in enumerate(info["dets"]):
        print(f"   stage-1 mask {k:2d} ({d.sum() / (H * W):5.1%} of the frame): {v}")
n_ignored = sum(v.startswith("not") for _, _, info in checks_log for _, v in info.get("dets", []))
say(f"Over all checks: {n_ignored} unmatched stage-1 masks were ignored for covering < 10% of the untracked area on their own")

## 6. Pass 2, "offline tracking": start again, cleanly (paper pp. 22–23, Eq. 14)

**Why a second pass?** In pass 1 an object that appeared at frame 50 is only registered at the next check, say frame 80. The paper (p. 22) says the offline pass fixes these "delayed initialization and incomplete trajectories prior to its registration".

**What the code does:** reset SAM2. Give every object its mask at its *registration* frame. Propagate **forward from frame 0** in one go, with no breaks.

**Can a forward pass give an object a mask *before* the frame it was given at?** In SAM2, when frame t is tracked, memory attention uses **all** of an object's prompted frames, *including future ones* (`max_cond_frames_in_attn = -1` in `sam2_base.py`). So it is possible. Object *pointers* from the future are excluded (`only_obj_ptrs_in_the_past_for_eval: true`). Earlier in our chat I guessed it would not fill those frames (my "P39"); here we measure it.

We also compare **mask coverage** (share of pixels covered by some object, averaged over frames) for pass 1 and pass 2. The paper reports 0.435 → 0.486 on 100 videos (p. 23).

In [ ]:
t = timer()
predictor.reset_state(state)
for i, (f, m) in first_appearance.items():
    predictor.add_new_mask(state, f, i, m.astype(np.uint8))
pass2 = {}
for f, tracked in propagate(state, 0, T - 1):
    pass2[f] = {i: P.encode_rle(m) for i, m in tracked.items() if m.any()}
say(f"Pass 2 done ({t()})")

def coverage(per_frame):
    cov = []
    for f in range(T):
        u = np.zeros((H, W), bool)
        for r in per_frame.get(f, {}).values():
            u |= P.decode_rle(r)
        cov.append(u.mean())
    return np.array(cov)
cov1, cov2 = coverage(pass1), coverage(pass2)
say(f"Mean mask coverage: pass 1 only {cov1.mean():.3f} → pass 1 + pass 2 {cov2.mean():.3f}  (paper, 100 videos: 0.435 → 0.486)")
plt.figure(figsize=(11, 2.8))
plt.plot(cov1, label="pass 1 (online)"); plt.plot(cov2, label="pass 2 (offline)")
for i, (f, _) in first_appearance.items():
    if f > seed:
        plt.axvline(f, c="k", lw=0.5)
plt.xlabel("frame"); plt.ylabel("covered share"); plt.legend(); plt.title("Mask coverage per frame (vertical lines = registrations)")
plt.savefig(f"{OUT}/6_coverage.png", dpi=70, bbox_inches="tight"); plt.show()

In [ ]:
late = [i for i, (f, _) in first_appearance.items() if f > seed]
backfill = {}
for i in late:
    f0 = first_appearance[i][0]
    before = [f for f in range(f0) if i in pass2[f]]
    backfill[i] = before
    say(f"Object {i}: registered at frame {f0}; pass 2 gives it a mask in {len(before)} of the {f0} earlier frames"
        + (f" (from frame {before[0]})" if before else ""))
if not late:
    say("No object was added after the first frame in this clip, so there is nothing to back-fill. Try MAX_FRAMES = None.")

fig, ax = plt.subplots(figsize=(12, 0.2 * next_id + 1.5))
for i in range(next_id):
    vis = [f for f in range(T) if i in pass2[f]]
    ax.scatter(vis, [i] * len(vis), s=5, marker="s", color=colour(i))
    ax.scatter([first_appearance[i][0]], [i], marker="|", s=200, c="k")
ax.invert_yaxis(); ax.set_xlabel("frame"); ax.set_ylabel("object id")
ax.set_title("Pass 2: frames in which each object has a mask. Black tick = frame where pass 1 registered it")
fig.savefig(f"{OUT}/6_timeline.png", dpi=70, bbox_inches="tight"); plt.show()

if late and any(backfill.values()):
    i = max(late, key=lambda k: len(backfill[k]))
    f0 = first_appearance[i][0]
    fs = sorted(set(np.linspace(backfill[i][0], f0 - 1, 4).astype(int))) + [f0]
    show([overlay(frames[f], {i: P.decode_rle(pass2[f][i])} if i in pass2[f] else {}) for f in fs],
         [f"frame {f}" + (" (registered here)" if f == f0 else " (before registration)") for f in fs],
         cols=len(fs), size=3.5, save="6_backfill_example.png",
         suptitle=f"Object {i}: masks pass 2 produced BEFORE the frame it was given at. Are they right?")

In [ ]:
idx = np.linspace(0, T - 1, 6).astype(int)
show([overlay(frames[f], {i: P.decode_rle(r) for i, r in pass2[f].items()}) for f in idx],
     [f"frame {f}: {len(pass2[f])} objects" for f in idx], save="6_final_tracks.png",
     suptitle="Stage 2 output: every object keeps ONE id (and colour) through time")

## 7. Check: their `stage2_track` on the same input should give the same tracks

We give their function the same stage-1 masks. Frames we skipped in stage 1 get an empty placeholder, which their code never reads. We also give it the same loaded SAM2. Same registration frames and mask IoU ≈ 1 means our hand-written version is their algorithm.

In [ ]:
if RUN_OFFICIAL_CHECK:
    t = timer()
    del state; torch.cuda.empty_cache()
    pcfg = P.PipelineConfig(); pcfg.tracking = tcfg; pcfg.io.video_path = CLIP_DIR
    masks_artifact = {"height": H, "width": W, "total_frames": T, "frame_sample_rate": RATE,
                      "sampled_frame_indices": sampled,
                      "frames": [[P.encode_rle(m) for m in stage1[f]] if f in stage1 else [P.empty_rle(H, W)] for f in sampled]}
    tracker = P.VideoTracker.__new__(P.VideoTracker)            # their wrapper, around the predictor already loaded
    tracker.predictor, tracker.device, tracker.offload = predictor, dev, tcfg.offload_to_cpu
    official = P.stage2_track(pcfg, masks_artifact, CLIP_DIR, tracker)
    same_first = len(official["objects"]) == next_id and all(
        o["first_frame"] == first_appearance[o["object_id"]][0] for o in official["objects"])
    ious = []
    for o in official["objects"]:
        for f, r in enumerate(o["masks"]):
            ours = pass2[f].get(o["object_id"])
            if r is None and ours is None:
                continue
            a = P.decode_rle(r) if r else np.zeros((H, W), bool)
            b = P.decode_rle(ours) if ours else np.zeros((H, W), bool)
            ious.append((a & b).sum() / max((a | b).sum(), 1))
    say(f"Official stage2_track: {len(official['objects'])} objects (ours {next_id}); same registration frames: {same_first}; "
        f"mask IoU ours vs theirs: mean {np.mean(ious):.4f}, min {np.min(ious):.4f} ({t()})")

## 8. Save

- The tracks go to `data/stage12/` (gitignored). They are the input for stage 3 (cleanup) next time.
- The pictures, a GIF and `README.md` with all the numbers go to `notes/stage12/`. Push these.

In [ ]:
from PIL import Image
from IPython.display import Image as IPImage, display
name = os.path.splitext(os.path.basename(VIDEO))[0]
tracks = {"video": VIDEO, "fps": fps, "height": H, "width": W, "total_frames": T,
          "objects": [{"object_id": i, "first_frame": first_appearance[i][0],
                       "masks": [pass2[f].get(i) for f in range(T)]} for i in range(next_id)]}
os.makedirs("data/stage12", exist_ok=True)
with open(f"data/stage12/{name}_first{T}_tracks.json", "w") as fh:
    json.dump(tracks, fh)

gif = []
for f in range(0, T, 3):
    im = overlay(frames[f], {i: P.decode_rle(r) for i, r in pass2[f].items()})
    gif.append(Image.fromarray(cv2.resize(im, (W // 2, H // 2))))
gif[0].save(f"{OUT}/tracks.gif", save_all=True, append_images=gif[1:], duration=int(3000 / fps), loop=0)
display(IPImage(filename=f"{OUT}/tracks.gif"))

imgs = sorted(p for p in os.listdir(OUT) if p.endswith((".png", ".gif")))
with open(f"{OUT}/README.md", "w") as fh:
    fh.write(f"# Stages 1–2 by hand: `{name}`, first {T} frames\n\n" + "\n\n".join(LOG) + "\n\n"
             + "\n".join(f"![{p}]({p})" for p in imgs) + "\n")
print("saved", OUT, "and data/stage12/")
print("now in a terminal:  git add notes/stage12 && git commit -m 'stages 1-2 by hand' && git push")

## 9. What to look for (bring these back to the chat)

1. **Section 2:** for a 1,000-frame video, how far apart are the re-discovery checks in seconds? What happens to an object that enters and leaves between two checks?
2. **Section 5:** were any stage-1 masks *orange* (unmatched but ignored as too small)? What were they? Under the paper's Eq. 11 they would have counted.
3. **Section 5:** were some *red* "new" objects really parts of objects already tracked (a hand, when the person is tracked)? Matching needs ≥ 90% inside one tracked object, so a partly-visible part becomes a new object. Stage 3 (cleanup) is meant to remove these.
4. **Section 6:** did pass 2 fill frames before registration, and do those masks look right?
5. **Section 6:** pass-1 vs pass-2 coverage against the paper's 0.435 → 0.486.
6. **Timeline:** any ids that vanish and come back? Any objects that end up covering the same thing?